# செலவு கோரிக்கை பகுப்பாய்வு

இந்த நோட்புக் பயணச் செலவுகளுக்கான உள்ளூர் ரசீத்ச் சித்திரங்களிலிருந்து தகவல்களைக் கொண்டு செயல்படும் பிளக்கின்களை பயன்படுத்தி முகவர்கள் எப்படி உருவாக்கப்படுகிறார்கள் என்பதை, செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குவது மற்றும் ஒரு பட்டாச்சார்ட் மூலம் செலவு தரவுகளை காட்சிப்படுத்துவது ஆகியவற்றை விளக்குகிறது. முகவர்கள் செயல் சூழலைப் பொருத்து செயல்பாடுகளை தானாகத் தேர்ந்தெடுக்கிறார்கள்.

படிகள்:
1. OCR முகவர் உள்ளூர் ரசீத்ச் சித்திரத்தை செயலாக்கி பயணச் செலவு தரவுகளை எடுக்கிறார்.
2. மின்னஞ்சல் முகவர் ஒரு செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறார்.

### ஒரு பயணச் செலவு சூழல் உதாரணம்:
நீங்கள் வேறுநகரில் உள்ள ஒரு வணிக கூட்டத்திற்காக பயணிக்கும் ஊழியராக இருக்கிறீர்கள் எனக் கற்பனை செய்யுங்கள். உங்கள் நிறுவனம் அனைத்து நியாயமான பயணச் செலவுகளையும் மீட்டெடுக்கும் கொள்கையை கொண்டுள்ளது. இதோ சாத்தியமான பயணச் செலவுகளின் விரிவுரை:
- போக்குவரத்து:
உங்கள் வீட்டுத்திகழிருந்து இலக்குநகரம் வரை சுற்றுலா விமானப் பயணம்.
விமான நிலையத்திற்கு மற்றும் விமான நிலையத்திலிருந்து டாக்சி அல்லது ஓட்டுநர் சேவைகள்.
இலக்குநகரத்தின் உள்ளாட்சி போக்குவரத்து (பொது போக்குவரத்து, வாடகை கார்கள் அல்லது டாக்சிகள் போன்றவை).

- தங்குமிடம்:
கூட்டத் தளத்தின் அருகில் இருக்கும் மிதமான வணிக ஹோட்டலில் மூன்று இரவு தங்கல்.

- உணவுகள்:
நிறுவனத்தின் தினசரி உணவு கொள்கையின் அடிப்படையில் காலை உணவு, மதிய உணவு மற்றும் இரவு உணவுக்கான தினசரி கொடுப்பனவு.

- பல்வேறு செலவுகள்:
விமான நிலையத்தில் கார் நிறுத்தும் கட்டணங்கள்.
ஹோட்டலில் இணைய அணுகல் கட்டணங்கள்.
கூர்மையான சேவை அல்லது சிறிய கட்டணங்கள்.

- ஆவணம்:
நீங்கள் அனைத்து ரசீத்கள் (விமானங்கள், டாக்சிகள், ஹோட்டல், உணவுகள் மற்றும் மற்றவை) மற்றும் செலவு அறிக்கையை பரிசோதனைக்கு சமர்ப்பிக்கிறீர்கள்.


## தேவையான நூலகங்களை இறக்குமதி செய்க

நோட்புக்குக்கான தேவையான நூலகங்களை மற்றும் மாட்யூலை இறக்குமதி செய்க.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## செலவுக் மாடல்களை வகுத்தமைக்கவும்

 தனிப்பட்ட செலவுகளுக்கான பைடான்டிக் மாதிரியை உருவாக்கவும், மற்றும் பயனர் கேள்வியைக் கட்டமைக்கப்பட்ட செலவு தரவாக மாற்ற ExpenseFormatter வகுப்பை உருவாக்கவும்.

 ஒவ்வொரு செலவும் கீழ்க்காணும் வடிவத்தில் பிரதிபலிக்கப்படும்:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## கருவிகளை வரையறுத்தல் - மின்னஞ்சல் உருவாக்குதல்

ஒரு செலவுக் குறைப்பு சமர்ப்பிக்க மின்னஞ்சலை உருவாக்க ஒரு கருவி செயல்பாட்டை உருவாக்குங்கள்.
- இந்த கருவி Microsoft முகவர் கட்டமைப்பின் `@tool` அலங்கரிப்பை பயன்படுத்துகிறது.
- இது செலவுகளின் மொத்த எண்ணிக்கையை கணக்கிட்டு விவரங்களை மின்னஞ்சல் உடலில் வடிவமைக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ரசீது படங்களில் இருந்து பயணச் செலவுகளை எடுக்கும் கருவி

ரசீது படங்களிலிருந்து பயணச் செலவுகளை எடுக்க ஒரு கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இல் இருந்து `@tool` அலங்கரிப்பாளரைப் பயன்படுத்துகிறது.
- இது ரசீது படத்தைப் படித்துக்கொண்டு, அதை base64 ஆக குறியாக்கம் செய்து, முகவரியால் பகுப்பாய்வு செய்ய தரவு URI ஐ வழங்குகிறது.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

Define the agents and wire them into a sequential workflow using `WorkflowBuilder`.
- The OCR agent extracts structured expense data from the receipt image using the `load_receipt_image` tool.
- The Email agent takes the extracted data and generates a professional expense claim email using the `generate_expense_email` tool.
- `WorkflowBuilder` with `add_edge` creates a sequential pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## முக்கிய செயல்பாடு

தொடர்ச்சியான பணிசேகரியை உருவாக்கி அதை இயக்கி ரசீது படத்தை செயல்படுத்தவும் மற்றும் செலவு கோரிக்கை மின்னஞ்சலை உருவாக்கவும்.


> **குறிப்பு:** இந்த வேலைநடை தற்போது ரசீது படத்தை base64 உரையாக கடத்துகிறது, இதனை பெரும்பாலான உரையாடல் மாதிரிகள் (gpt-5-mini உட்பட) படம் என கருதாது.
> இது மாதிரி கருத்து சாளரத்தை தாண்டக்கூடும். Azure AI Vision (அதேபோல் வேறு OCR கருவியோடு) OCR இயக்கி மட்டும் எடுத்து பிடிக்கப்பட்ட உரையை அனுப்ப அல்லது படத்தை `image_url` செய்தி என மாற்றி அனுப்ப விரும்புவது நல்லது.
> கருத்து பிழைகளை தவிர்க்க மட்டுமே வேண்டும் என்றால், சிறிய ரசீது படம் அல்லது பெரிய கருத்து சாளரத்துடன் கூடிய மாதிரியை முயற்சிக்கவும்.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
